<style type="text/css">.tg  {border-collapse:collapse;border-spacing:0;}.tg td{border-color:rgb(16, 137, 182);border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;overflow:hidden;padding:10px 5px;word-break:normal;}.tg th{border-color:rgb(16, 137, 182);border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}.tg .tg-73oq{border-color:rgb(10, 89, 162);text-align:left;vertical-align:middle}.tg .tg-42lt{border-color:rgb(10, 89, 162);text-align:center;vertical-align:middle}.tg .tg-5qt9{font-size:small;text-align:left;vertical-align:middle}</style><table class="tg"><thead>  <tr>    <th class="tg-73oq"><img src="https://raw.githubusercontent.com/euroargodev/argopy/master/docs/_static/argopy_logo_long.png" alt="Argopy logo" width="120" height="60"></th>    <th class='tg-42lt'><h1>Argo referencing tools (vocabulary)</h1></th>  </tr></thead><tbody>  <tr>    <td class="tg-5qt9" colspan="2"><span style="font-weight:bold">Author :</span> <a href="//annuaire.ifremer.fr/cv/17182" target="_blank" rel="noopener noreferrer">G. Maze</a></td>  </tr>  <tr>    <td class="tg-5qt9" colspan="2">🏷️ This notebook is compatible with Argopy versions &gt;= <a href="https://argopy.readthedocs.io/en/v1.5.0" target="_blank" rel="noopener noreferrer">v1.5.0</a></td>  </tr>  <tr>    <td class="tg-5qt9" colspan="2">© <a href="https://github.com/euroargodev/argopy-training/blob/main/LICENSE" target="_blank" rel="noopener noreferrer">European Union Public Licence (EUPL) v1.2</a>, see at the bottom of this notebook for more.</td>  </tr></tbody></table>
**Description:**

This notebook will show how to access the Argo reference values and tables:
- How to load one reference value,
- How to load one reference table,
- Look for a reference table,

Argo reference tables/values are used to enforce a controlled vocabulary for Argo meta-data content and parameter attributes, like the description of quality control flags or sensor names.

The [dedicated Argopy documentation page is here (↗)](https://argopy.readthedocs.io/en/v1.5.0/advanced-tools/metadata/nvs_ref_tables.html).


**Table of Contents**
- [ArgoReferenceValue](#argoreferencevalue)
  - [Creation](#creation)
    - [Disambiguation](#disambiguation)
    - [🔍 Pro tip](#🔍-pro-tip)
  - [Attributes](#attributes)
    - [🔍 Pro tip](#🔍-pro-tip)
  - [Extra attributes for values of R03, R14, R18 tables](#extra-attributes-for-values-of-r03,-r14,-r18-tables)
  - [Export methods](#export-methods)
- [ArgoReferenceTable](#argoreferencetable)
  - [Creation](#creation)
  - [Supported tables ?](#supported-tables-?)
  - [Attributes](#attributes)
    - [🔍 Pro tip](#🔍-pro-tip)
  - [Indexing and values](#indexing-and-values)
    - [✏️ EXERCISE](#✏️-exercise)
  - [Export methods](#export-methods)
    - [🔍 Pro tip](#🔍-pro-tip)
  - [Search method](#search-method)
    - [✏️ EXERCISE](#✏️-exercise)
    - [🔍 Pro tip](#🔍-pro-tip)
***

## ArgoReferenceValue

The [ArgoReferenceValue (↗)](https://argopy.readthedocs.io/en/v1.5.0/advanced-tools/metadata/nvs_ref_tables.html#reference-values) instance holds all the Argo referencing system information:

- the comprehensive logic (e.g. the reference table is automatically determined when possible, hints are return otherwise),
- the value meta-data are in read-only attributes, e.g. ``arv.definition`` and ``arv.version``, ``arv.deprecated``, etc..., 
- value meta-data export methods, e.g. ``arv.to_dict()``, ``arv.to_json()``.

### Creation

To create an Argo reference value instance, you can use its name, as it is in an Argo netcdf file:

In [ ]:
from argopy import ArgoReferenceValue

# One possible value for the Argo parameter 'SENSOR_MODEL':
arv = ArgoReferenceValue('BBP700')
arv

\
but you can also use the NVS/URN jargon with the ``from_urn`` method:

In [ ]:
arv = ArgoReferenceValue.from_urn('SDN:R03::BBP700')

#### Disambiguation

Note that for ambiguous value seen in more than one Reference Table, you need to specify the reference table to consider.

For instance ``4`` is a reference value showing up 5 tables (eg: ``RTQC_TESTID`` or ``RT_QC_FLAG``):

In [ ]:
arv = ArgoReferenceValue('4', reference='RTQC_TESTID')
arv

In [ ]:
arv = ArgoReferenceValue('4', reference='RT_QC_FLAG')
arv

#### 🔍 Pro tip

For ambiguous values, Argopy will provide the list of all tables where the value is found to help better select the appropriate one:

In [ ]:
try:
    ArgoReferenceValue('4')
except Exception as e:
    print(e)

### Attributes

In [ ]:
# All possible attributes are listed in:
arv.attrs

\
The probably most interesting reference value attributes are:

In [ ]:
arv.name       # Term-id of the URN, eg 'AANDERAA_OPTODE_3835'

In [ ]:
arv.long_name  # nvs["skos:prefLabel"]["@value"]

In [ ]:
arv.definition # nvs["skos:definition"]["@value"]

In [ ]:
arv.deprecated # nvs["owl:deprecated"]

In [ ]:
arv.parameter  # The netcdf parameter this concept applies to (eg 'SENSOR_MODEL')

In [ ]:
arv.reference  # The reference table this concept belongs to, can be used on a ArgoReferenceTable (eg 'R27')

\
but more technical attributes are also available:

In [ ]:
arv.version    # nvs["owl:versionInfo"]

In [ ]:
arv.date       # nvs["dc:date"]

In [ ]:
arv.uri        # nvs["@id"]

In [ ]:
arv.urn        # nvs["skos:notation"]

#### 🔍 Pro tip

For experts, the relationships with other Reference Values or Context are also available with:

In [ ]:
arv.broader    # nvs["skos:broader"]
arv.narrower   # nvs["skos:narrower"]
arv.related    # nvs["skos:related"]
arv.sameas     # nvs["owl:sameAs"]
arv.context    # nvs["@context"]

### Extra attributes for values of R03, R14, R18 tables

Sometime, the reference value definition is formatted to hold extra-attributes.

Argopy automatically infers those extra-attributes like Local_Attributes and Properties, and make them available as regular attributes to the ArgoReferenceValue instance.

\
For instance, any values from R03 table:

In [ ]:
arv = ArgoReferenceValue('BBP470')

will offer these extra-attributes:

In [ ]:
arv.extra

In [ ]:
arv.extra['Local_Attributes'].long_name

In [ ]:
arv.extra['Properties'].category

\
For values from R14 table:

In [ ]:
arv = ArgoReferenceValue('T000015')
arv.extra

In [ ]:
arv.extra['Template_Values'].unit

\
and for values from R18 table:

In [ ]:
arv = ArgoReferenceValue('CB00001')
arv.extra

In [ ]:
arv.extra['Template_Values'].short_sensor_name

### Export methods

You can export all value attributes to a dictionary:

In [ ]:
arv = ArgoReferenceValue('AANDERAA_OPTODE_3835')
arv.to_dict()

\
but you can also select attributes to export:

In [ ]:
arv.to_dict(keys=['name', 'deprecated'])

\
At last, you can export to a json structure as well:

In [ ]:
arv.to_json()  # In memory
# arv.to_json('reference_value.json')  # To a json file
# arv.to_json('reference_value.json', keys=['name', 'deprecated'])  # Select attributes to export

## ArgoReferenceTable

The [ArgoReferenceTable (↗)](https://argopy.readthedocs.io/en/v1.5.0/advanced-tools/metadata/nvs_ref_tables.html#reference-tables) class holds all the Argo referencing system information:

- the comprehensive logic (use ``SENSOR`` in place of ``R25``),
- the table meta-data are in read-only attributes, e.g. ``art.description`` and ``art.version``, ``art.date``, etc...,
- the list of table values/rows, which are accessible through label-based indexing, e.g. ``art['CTD_TEMP_CNDC']``; this means that the instance can be used almost like a dictionary (e.g. for inclusion assertion and iteration),
- values export/search methods, e.g. ``art.search()``, ``art.to_dataframe()``,  ``art.to_dict()``

### Creation

In [ ]:
from argopy import ArgoReferenceTable

# Use an Argo parameter name, documented by one of the Argo reference tables:
art = ArgoReferenceTable('SENSOR')

# or a reference table identifier:
art = ArgoReferenceTable('R25')

# or a URN:
art = ArgoReferenceTable.from_urn('SDN:R25::CTD_TEMP')

art

### Supported tables ?

You can get a dictionary of all supported tables with the class ``valid_identifiers()`` method:

In [ ]:
ArgoReferenceTable.valid_identifiers()

### Attributes


In [ ]:
# All possible attributes are listed in:
art.attrs

In [ ]:
# Reference Table attributes:
art.parameter   # Name of the netcdf dataset parameter filled with values from this table
art.identifier  # Reference Table ID
art.description # [nvs['@graph']['@type']=='skos:Collection']["dc:description"]
art.uri         # [nvs['@graph']['@type']=='skos:Collection']["@id"]
art.version     # [nvs['@graph']['@type']=='skos:Collection']['owl:versionInfo']
art.date        # [nvs['@graph']['@type']=='skos:Collection']['dc:date']

#### 🔍 Pro tip

Raw NVS json data are available in the ``nvs`` attribute:

In [ ]:
art.nvs.keys()

### Indexing and values

A reference table is primarily a placeholder for a collection of *reference values*. The reference table class was designed to behave almost like a regular dictionary to access these [values (↗)](https://argopy.readthedocs.io/en/v1.5.0/generated/argopy.ArgoReferenceValue.html).

Let's show this with the SENSOR table:

In [ ]:
art = ArgoReferenceTable('SENSOR')
art

<br>
The number of reference values in the table is available from its length:

In [ ]:
len(art)     

<br>
The list of reference value names (or concept) within this reference table is availabe as keys:

In [ ]:
art.keys()

\
and the list of actual reference value content is available as a list of [value objects (↗)](https://argopy.readthedocs.io/en/v1.5.0/generated/argopy.ArgoReferenceValue.html).

Since this will render a long print, we just show the first 2 values of the list:

In [ ]:
art.values()[0:2]  # List of :class:`ArgoReferenceValue` instance

<br>
Since the table behaves like a dictionnary, you can check if a value key is present in the table:

In [ ]:
'CTD_TEMP_CNDC' in art  # Return True

<br>
and the table can be indexed by value key, like a simple dictionary:

In [ ]:
art['CTD_TEMP_CNDC']  # Return a :class:`ArgoReferenceValue` instance

#### ✏️ EXERCISE

The class also allows to iterate over all values/concepts with a for loop.

Iterate over all table reference values to pretty print all value long name.

In [ ]:
# Your code here

### Export methods

Export table attributes to a dictionary (rq: this is not to export values):

In [ ]:
art.to_dict()

#### 🔍 Pro tip
You can select the table attributes to export:

In [ ]:
art.to_dict(keys=['parameter', 'date', 'uri'])

\
Last, it is possible to export [table values (↗)](https://argopy.readthedocs.io/en/v1.5.0/generated/argopy.ArgoReferenceValue.html) as a pandas DataFrame with:

```python
art.to_dataframe()
```

Note that you can select the value attributes to export in the dataframe columns:

In [ ]:
art.to_dataframe(columns=['name', 'deprecated', 'uri'])

### Search method

For large tables, or simply to explore table content without a precise a priori knowledge, we provide a search method on any of the reference values attributes.

Let's show this with the SENSOR_MODEL reference table:

In [ ]:
art = ArgoReferenceTable('SENSOR_MODEL')
art

\
The search methods return a list of [ArgoReferenceValue (↗)](https://argopy.readthedocs.io/en/v1.5.0/generated/argopy.ArgoReferenceValue.html) with match:

In [ ]:
art.search(name='RAMSES')  # Search in values name

In [ ]:
art.search(definition='imaging')  # Search in values definition

#### ✏️ EXERCISE

Search in values long name for any TriOS sensor.

In [ ]:
# Your code here

#### 🔍 Pro tip

It is possible to change the search output format to a Pandas dataframe:

In [ ]:
art.search(long_name='TriOS', output='df')


## 🏁 End of the notebook

***
#### 👀 Useful argopy commands
```python
argopy.reset_options()
argopy.show_options()
argopy.status()
argopy.clear_cache()
argopy.show_versions()
```
#### ⚖️ License Information
This Jupyter Notebook is licensed under the **European Union Public Licence (EUPL) v1.2**.

| Permissions      | Limitations     | Conditions                     |
|------------------|-----------------|--------------------------------|
| ✔ Commercial use | ❌ Liability     | ⓘ License and copyright notice |
| ✔ Modification   | ❌ Trademark use | ⓘ Disclose source              |
| ✔ Distribution   | ❌ Warranty      | ⓘ State changes                |
| ✔ Patent use     |                  | ⓘ Network use is distribution  |
| ✔ Private use    |                  | ⓘ Same license                 |

For more details, visit: [EUPL v1.2 Full Text (↗)](https://github.com/euroargodev/argopy-training/blob/main/LICENSE).

#### 🤝 Sponsor
![logo (↗)](https://raw.githubusercontent.com/euroargodev/argopy-training/refs/heads/main/for_nb_producers/disclaimer_argopy_EAONE.png)
***
